In [ ]:
# =======================
# FUNCIONES AUXILIARES
# =======================
import os, time, math,glob
import numpy as np
import pandas as pd
from pathlib import Path
import matplotlib.pyplot as plt
import warnings, sys, contextlib, functools

# Rutas relativas a la carpeta actual (cwd)
BASE_PATH = os.path.join(os.getcwd(), 'fraud_stream_parquet')
OUT_DIR = os.path.join(os.getcwd(), 'baseline1_results')

# Crear directorio de salida si no existe
os.makedirs(OUT_DIR, exist_ok=True)

# -----------------------------
# Silenciar salida verbose / barras de progreso / warnings
# -----------------------------
# 1) Intentar desactivar barras de progreso de tqdm globalmente (la mayoría de librerías usan tqdm or tqdm.auto)
try:
    import tqdm
    try:
        import tqdm.auto as tqdm_auto
    except Exception:
        tqdm_auto = None
    tqdm.tqdm = functools.partial(tqdm.tqdm, disable=True)
    if tqdm_auto is not None:
        tqdm_auto.tqdm = functools.partial(tqdm_auto.tqdm, disable=True)
except Exception:
    pass

# 2) Ignorar DeprecationWarning (p. ej. avisos de avalanche/MIR que llenan la salida)
warnings.filterwarnings('ignore', category=DeprecationWarning)

# 3) Context manager para suprimir stdout/stderr alrededor de llamadas ruidosas (ej. strategy.train)
@contextlib.contextmanager
def suppress_output(suppress_stdout=True, suppress_stderr=True):
    """Redirige temporalmente stdout/stderr a os.devnull."""
    old_stdout, old_stderr = sys.stdout, sys.stderr
    devnull_streams = []
    try:
        if suppress_stdout:
            sys.stdout = open(os.devnull, 'w')
            devnull_streams.append(sys.stdout)
        if suppress_stderr:
            sys.stderr = open(os.devnull, 'w')
            devnull_streams.append(sys.stderr)
        yield
    finally:
        for s in devnull_streams:
            try:
                s.close()
            except Exception:
                pass
        sys.stdout, sys.stderr = old_stdout, old_stderr

FEATURES = [
    'TX_AMOUNT', 'TX_TIME_DAYS', 'TX_TIME_SECONDS',
    'x_customer_id','y_customer_id','mean_amount','std_amount','mean_nb_tx_per_day',
    'x_terminal_id','y_terminal_id'
]
TARGET = 'TX_FRAUD'

PRE_MONTHS = 4                 # meses para pretraining
VAL_DAYS_LAST_MONTH = 14       # días del mes 4 para calibrar umbral F1
START_DATE  = "2025-01-01" 
GRANULARITY = 'week'          # 'month' (recomendado). Si quieres diario: 'day'.
TIMELINE_FILE = os.path.join(BASE_PATH, 'timeline.parquet')  # opcional; si no existe, TTA90 usa un solo régimen

# TTA90: ventanas para "plateau" por régimen (si GRANULARITY='month', usa 2; si 'day', usa 10)
PLATEAU_WINDOW = 4 #2 if GRANULARITY == 'month' else 10
TTA_TARGET_FRAC = 0.90  # 90%

def load_all_parquet_by_year(base_path: str) -> pd.DataFrame:
    year_dirs = sorted(glob.glob(os.path.join(base_path, "TX_YEAR=*")))
    dfs = []
    for ydir in year_dirs:
        files = sorted(glob.glob(os.path.join(ydir, "*.parquet")))
        if not files:
            continue
        dfs.append(pd.concat([pd.read_parquet(f) for f in files], ignore_index=True))
    if not dfs:
        raise RuntimeError(f"No se encontraron Parquet en {base_path}")
    df = pd.concat(dfs, ignore_index=True)
    # Orden temporal sin TX_DATETIME
    df = df.sort_values(['TX_YEAR','TX_MONTH','TX_DAY','TX_TIME_SECONDS'], kind='mergesort').reset_index(drop=True)
    #print(df.head())
   # print(df.columns.tolist())
    return df

def add_time_indexes(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    first_year = int(df['TX_YEAR'].min())
    df['_month_idx'] = (df['TX_YEAR'] - first_year) * 12 + df['TX_MONTH']  # 1..24

    df['_week_idx']  = (df['TX_TIME_DAYS'] // 7).astype(int) + 1  
  
    df['_day_abs'] = (df['_month_idx'] - 1) * 31 + df['TX_DAY'] 
    return df

def split_pretrain_val(df: pd.DataFrame, pre_months=4, val_days_last_month=14):
    df = df.copy()
    m = pre_months
    pre_mask = (df['_month_idx'] < m)              # meses 1..3
    last_month_mask = (df['_month_idx'] == m)      # mes 4
    # últimas 2 semanas del mes 4 como val
    dmax = int(df.loc[last_month_mask, 'TX_DAY'].max())
    cutoff = max(1, dmax - val_days_last_month + 1)
    train_last = last_month_mask & (df['TX_DAY'] < cutoff)
    val_last   = last_month_mask & (df['TX_DAY'] >= cutoff)
    train_mask = pre_mask | train_last
    val_mask   = val_last
    return train_mask, val_mask

def compute_base_week(eval_df: pd.DataFrame) -> int:
    base_week = int(eval_df['TX_TIME_DAYS'].min() // 7) + 1
    base_day = int(eval_df['TX_TIME_DAYS'].min()) 
    base_date = pd.to_datetime(START_DATE) + pd.Timedelta(days=base_day)
    print(f"Semana base: {base_week}, Fecha: {base_date.strftime('%Y-%m-%d')}")
    # primera semana presente en el df de evaluación
    return base_week

def compute_month_markers(eval_df: pd.DataFrame, base_week: int):
    marks = []
    months = (eval_df[['TX_YEAR','TX_MONTH']]
              .drop_duplicates()
              .sort_values(['TX_YEAR','TX_MONTH']))
    for _, r in months.iterrows():
        y, m = int(r['TX_YEAR']), int(r['TX_MONTH'])
        day_min = int(eval_df[(eval_df['TX_YEAR']==y) & (eval_df['TX_MONTH']==m)]['TX_TIME_DAYS'].min())
        wk_abs  = day_min // 7 + 1
        wk_rel  = int(wk_abs - base_week + 1)
        marks.append((wk_rel, f"{y}-{m:02d}"))
    return marks

def compute_bimonth_markers(eval_df: pd.DataFrame, pre_months: int, base_week: int):
    # índices de mes relativos al primer año del eval_df
    fy = int(eval_df['TX_YEAR'].min())
    eval_df = eval_df.copy()
    eval_df['_midx'] = (eval_df['TX_YEAR'] - fy) * 12 + eval_df['TX_MONTH'] 

    start_m = int(eval_df['_midx'].min()) 
    end_m   = int(eval_df['_midx'].max())

    
    M0 = start_m
    boundaries = list(range(M0 + 2, end_m + 1, 2))

    marks = []
    for b in boundaries:
        y = fy + (b - 1) // 12
        m = (b - 1) % 12 + 1
        # 1er día (en eval_df) de ese mes b
        rows = eval_df[(eval_df['TX_YEAR']==y) & (eval_df['TX_MONTH']==m)]
        if rows.empty:
            continue
        day_min = int(rows['TX_TIME_DAYS'].min())
        wk_abs  = day_min // 7 + 1
        wk_rel  = int(wk_abs - base_week + 1)
        marks.append((wk_rel, f"M{m:02d}"))
    return marks

def _shade_bimonth_segments(ax, x_vals, bimonth_markers, start_cycle_index=0):
    """
    Pinta franjas desde el inicio (x_min) hasta el final (x_max),
    alternando 3 colores por cada bimestre. Colorea también el 1er tramo.
    """
    import numpy as np
    if not bimonth_markers:
        return
    x_min = float(np.nanmin(x_vals))
    x_max = float(np.nanmax(x_vals))
    bounds = [x_min] + sorted([wk for wk, _ in bimonth_markers]) + [x_max + 1e-9]

    cycle = ["#9fbff2", "#f6e296", "#b7f7bb"]  # azul claro, ámbar, verde claro
    cidx = start_cycle_index % len(cycle)

    for i in range(len(bounds) - 1):
        left, right = bounds[i], bounds[i + 1]
        ax.axvspan(left, right, facecolor=cycle[cidx], alpha=0.22, linewidth=0)
        cidx = (cidx + 1) % len(cycle)


def _calculate_plateau_series(metrics_df, seg_plateaux_df):
    """
    Transforma la tabla de segmentos (que tiene el plateau por bloque) en una
    serie continua (línea de escalones) que coincide con el índice semanal.
    """
    full_index = metrics_df['pos'].values

    plateau_series = pd.Series(index=full_index, dtype=float)
    

    for _, segment in seg_plateaux_df.iterrows():
        s, e = int(segment['start_pos']), int(segment['end_pos'])
        P_loc = segment['plateau_local']

        plateau_series.loc[s:e] = P_loc
        
    return plateau_series.sort_index()

def plot_series(metrics_df, train_times_m, out_dir, month_markers=None, bimonth_markers=None,title=None,seg_plateaux_df=None):
    x = metrics_df['pos']

    def _apply_month_ticks():
        if month_markers:
            for wk, _ in month_markers:
                plt.axvline(wk, color='gray', alpha=0.15)  # líneas suaves por mes
            xs, labs = zip(*month_markers)
            plt.xticks(xs, labs, rotation=45, ha='right')

    def _apply_bimonth_lines():
        if bimonth_markers:
            for wk, _ in bimonth_markers:
                plt.axvline(wk, color='black', linestyle=':', linewidth=1.2, alpha=0.7)  # punteada


     # === AUPRC semanal ===
    '''
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['AUPRC'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('AUPRC'); plt.title(f'AUPRC por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'auprc_semana_{title}.png')); plt.close()
    '''
 
    # === F1 semanal (umbral fijo) ===
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x,bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['F1'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('F1 (umbral fijo)'); plt.title(f'F1 por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'f1_semana_{title}.png')); plt.close()

        
    # === G-Mean semanal (umbral fijo) ===
    '''
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['G-Mean'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('G-Mean (umbral fijo)')
    plt.title(f'G-Mean por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'gmean_semana_{title}.png')); plt.close()
    '''
    # === Recall semanal (umbral fijo) ===
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['Recall'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('Recall (umbral fijo)')
    plt.title(f'Recall por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'recall_semana_{title}.png')); plt.close()

     # === Recall semanal (umbral fijo) ===
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=0)
    plt.plot(x, metrics_df['Precision'].values)
    plt.ylim(-0.05, 1.05)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('Precision (umbral fijo)')
    plt.title(f'Precision por semana ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'precision_semana_{title}.png')); plt.close()

     # === Latencia semanal ===
    plt.figure()
    ax = plt.gca()
    _shade_bimonth_segments(ax, x, bimonth_markers,start_cycle_index=0)
    plt.plot(x, metrics_df['infer_ms_per_tx'].values)
    _apply_month_ticks(); _apply_bimonth_lines()
    plt.xlabel('Semanas'); plt.ylabel('ms por chunck semanal'); plt.title(f'Latencia de inferencia ({title})')
    plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'latencia_semana_{title}.png')); plt.close()

    if train_times_m is not None and not train_times_m.empty:
        # === Tiempo de entrenamiento semanal ===
        plt.figure()
        ax = plt.gca()
        _shade_bimonth_segments(ax, x, bimonth_markers,start_cycle_index=0)
        plt.plot(x, train_times_m['train_time_s'].values)
        _apply_month_ticks(); _apply_bimonth_lines()
        plt.xlabel('Semanas'); plt.ylabel('Tiempo de entrenamiento (s)'); plt.title(f'Tiempo de entrenamiento ({title})')
        plt.tight_layout(); plt.savefig(os.path.join(out_dir, f'entrenamiento_semana_{title}.png')); plt.close()


def drawdown_series(metrics_df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula running max y drawdown% semana a semana sobre AUPRC.
    Devuelve df con columnas: pos, label, AUPRC, run_max, drawdown_pct.
    """
    out = metrics_df[['pos','label','AUPRC']].copy()
    a = out['AUPRC'].values
    run_max = np.maximum.accumulate(np.nan_to_num(a, nan=0.0))
    # dd% = (a - run_max)/run_max * 100 (negativo o 0). Si run_max==0 -> 0.
    dd = []
    for i in range(len(a)):
        rm = run_max[i]
        if rm <= 0:
            dd.append(0.0)
        else:
            dd.append( (a[i] - rm) / rm * 100.0 )
    out['run_max'] = run_max
    out['drawdown_pct'] = dd
    return out

def plot_drawdown_weekly(dd_df: pd.DataFrame, out_dir: str, month_markers=None, bimonth_markers=None,start_cycle_index=0,title=None):
    if dd_df is None or dd_df.empty:
        return
    
    x = dd_df['pos'].values
    y = dd_df['drawdown_pct'].values
    
    plt.figure()

    ax = plt.gca()

    # Franjas por bimestre (S1→S2→S3→…)
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=start_cycle_index)

    plt.plot(x, y)
    if month_markers:
        for wk, _ in month_markers:
            plt.axvline(wk, color='gray', alpha=0.15)
        xs, labs = zip(*month_markers)
        plt.xticks(xs, labs, rotation=45, ha='right')
    if bimonth_markers:
        for wk, _ in bimonth_markers:
            plt.axvline(wk, color='black', linestyle=':', linewidth=1.2, alpha=0.7)


    plt.xlabel('Semanas'); 
    plt.ylabel('Drawdown % (≤ 0)')
    plt.title(f'Olvido/estabilidad: Drawdown de AUPRC - {title}')
    plt.tight_layout(); 
    plt.savefig(os.path.join(out_dir, f'drawdown_semana_{title}.png')); plt.close()
    plt.close()

def schedule_to_week_segments_with_scenario(schedule_ym, eval_df, base_week):
    segs = []
    for step in schedule_ym:
        y1,m1 = int(step['start']['year']), int(step['start']['month'])
        y2,m2 = int(step['end']['year']),   int(step['end']['month'])
        scen  = int(step['scenario'])

        in_range = eval_df[
            ((eval_df['TX_YEAR'] >  y1) | ((eval_df['TX_YEAR']==y1) & (eval_df['TX_MONTH']>=m1))) &
            ((eval_df['TX_YEAR'] <  y2) | ((eval_df['TX_YEAR']==y2) & (eval_df['TX_MONTH']<=m2)))
        ]
        if in_range.empty:
            continue

        dmin, dmax = int(in_range['TX_TIME_DAYS'].min()), int(in_range['TX_TIME_DAYS'].max())
        wk1_abs, wk2_abs = dmin//7 + 1, dmax//7 + 1
        wk1 = int(wk1_abs - base_week + 1)
        wk2 = int(wk2_abs - base_week + 1)
        lab = f"{y1}-{m1:02d}..{y2}-{m2:02d} (S{scen})"
        segs.append({'start_pos': wk1, 'end_pos': wk2, 'label': lab, 'scenario': scen})

    return sorted(segs, key=lambda s: (s['start_pos'], s['end_pos']))

def _calc_plateau_from_tail(ap, window='auto', method='percentile', q=0.80):
    """
    Plateau = estadístico de cola de la serie AUPRC del segmento.
    window: entero o 'auto' (~30% del largo, min 3)
    method: 'mean' | 'median' | 'percentile'
    q: percentil (si method='percentile'), p.ej. 0.80
    """
    import numpy as np
    L = len(ap)
    W = max(3, int(round(0.3*L))) if (isinstance(window, str) and window.lower()=='auto') else int(window)
    W = max(1, min(W, L))
    tail = np.asarray(ap[-W:], dtype=float)

    if method == 'median':
        plateau = float(np.nanmedian(tail))
    elif method == 'percentile':
        plateau = float(np.nanpercentile(tail, q*100.0))
    else:
        plateau = float(np.nanmean(tail))
    return plateau, W


def compute_segment_plateaux(metrics_df, segments, window='auto', method='percentile', q=0.80):
    """
    Devuelve DF: [start_pos, end_pos, label, scenario, len_weeks, plateau_local]
    """
    import pandas as pd, numpy as np
    m = metrics_df.set_index('pos').sort_index()
    rows = []
    for seg in segments:
        s, e = int(seg['start_pos']), int(seg['end_pos'])
        sub = m.loc[m.index.intersection(range(s, e+1))]
        if sub.empty:
            rows.append({**seg, 'len_weeks': 0, 'plateau_local': np.nan})
            continue
        ap = sub['F1'].values
        plateau, _ = _calc_plateau_from_tail(ap, window=window, method=method, q=q)
        rows.append({**seg, 'len_weeks': int(len(ap)), 'plateau_local': float(plateau)})
    import pandas as pd
    return pd.DataFrame(rows).sort_values('start_pos').reset_index(drop=True)


def build_adaptation_series_with_blend(
    metrics_df, seg_df,
    ramp_weeks=4,               # duración de la rampa (semanas)
    ramp_start=0.0,             # inicio de rampa: float∈[0,1] (fracción del bloque) o int (semanas)
    cap=(0.0, 1.2),             # recorte final del índice para visual
    update_strategy='ema',      # 'ema' | 'replace' | 'rolling_k'
    alpha=0.3,                  # α para EMA de la memoria por escenario
    rolling_k=3,                # K para 'rolling_k'
    smooth='none',              # 'none' | 'mov3' | 'ema'
    ema_alpha=0.5,              # α para suavizado EMA del índice dentro del bloque
    clamp_to_local=None         # e.g., (0.85, 1.20) para acotar el índice respecto a límites fijos
):
   
    import numpy as np, pandas as pd, collections

    m = metrics_df.set_index('pos').sort_index()
    rows = []
    ref_plateau = {}  # memoria por escenario
    hist_plateaux = collections.defaultdict(list)

    for _, r in seg_df.sort_values('start_pos').iterrows():
        s, e   = int(r['start_pos']), int(r['end_pos'])
        scen   = int(r['scenario'])
        label  = r['label']
        P_loc  = float(r['plateau_local'])

        sub = m.loc[m.index.intersection(range(s, e+1))].copy()
        if sub.empty or not np.isfinite(P_loc) or P_loc <= 0:
            continue

        ap = sub['F1'].values.astype(float)
        L  = len(ap)

        # IAN clásico (local)
        idx_local = ap / P_loc

        # Estándar dinámico P_t (memoria -> actual) con rampa
        P_prev = ref_plateau.get(scen, np.nan)

        # Interpretar ramp_start (fracción o semanas)
        if isinstance(ramp_start, (int, np.integer)):
            t0 = max(0, int(ramp_start))
        else:
            # fracción del bloque [0..1]
            t0 = int(round(float(ramp_start) * L))
            t0 = min(max(0, t0), max(0, L-1))

        if np.isfinite(P_prev) and P_prev > 0:
            t = np.arange(L, dtype=float)
            # beta_t: 0 hasta t0; luego sube linealmente durante ramp_weeks
            if ramp_weeks is None or ramp_weeks <= 0:
                beta_t = (t >= t0).astype(float)  # salto brusco al llegar a t0
            else:
                beta_t = np.clip((t - t0) / float(ramp_weeks), 0.0, 1.0)
            P_t = (1.0 - beta_t) * P_prev + beta_t * P_loc
        else:
            P_t = np.full(L, P_loc, dtype=float)  # primera aparición del escenario

        idx_blend = ap / P_t

        # (opcional) clamp “rápido” a límites fijos (independiente del local)
        if clamp_to_local is not None:
            lo_c, hi_c = clamp_to_local
            idx_blend = np.clip(idx_blend, lo_c, hi_c)

        # Suavizado intra-bloque (sobre AdaptIdx_blend)
        if smooth == 'mov3' and L >= 3:
            # media móvil simple ventana 3
            kernel = np.ones(3, dtype=float) / 3.0
            idx_blend = np.convolve(idx_blend, kernel, mode='same')
        elif smooth == 'ema':
            out = np.empty_like(idx_blend)
            out[0] = idx_blend[0]
            a = float(ema_alpha)
            for i in range(1, L):
                out[i] = a * idx_blend[i] + (1.0 - a) * out[i-1]
            idx_blend = out

        # Cap visual final
        if cap is not None:
            lo, hi = cap
            if hi is None:
                idx_local = np.maximum(idx_local, lo)
                idx_blend = np.maximum(idx_blend, lo)
            else:
                idx_local = np.clip(idx_local, lo, hi)
                idx_blend = np.clip(idx_blend, lo, hi)

        sub = sub.reset_index()
        sub['segment']         = label
        sub['scenario']        = scen
        sub['AdaptIdx_local']  = idx_local
        sub['AdaptIdx_blend']  = idx_blend
        rows.append(sub[['pos','segment','scenario','AdaptIdx_local','AdaptIdx_blend']])

        # === Actualizar memoria del escenario ===
        if update_strategy == 'ema':
            if np.isfinite(P_prev) and P_prev > 0:
                P_post = (1.0 - alpha) * P_prev + alpha * P_loc
            else:
                P_post = P_loc
            ref_plateau[scen] = float(P_post)

        elif update_strategy == 'replace':
            ref_plateau[scen] = P_loc

        elif update_strategy == 'rolling_k':
            hist_plateaux[scen].append(P_loc)
            last_k = hist_plateaux[scen][-int(rolling_k):]
            ref_plateau[scen] = float(np.mean(last_k)) if len(last_k) > 0 else P_loc

        else:
            ref_plateau[scen] = P_loc  # fallback

    import pandas as pd
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame(
        columns=['pos','segment','scenario','AdaptIdx_local','AdaptIdx_blend']
    )


def plot_adapt_blend_line(adapt_df, out_dir, month_markers=None, bimonth_markers=None,
                          title="Índice de adaptación combinado (memoria → actual)",
                          break_segments=True, start_cycle_index=0):

    if adapt_df is None or adapt_df.empty:
        return

    # Ordenamos una sola vez y trazamos UNA sola línea (mismo color en todo)
    dfp = adapt_df.sort_values('pos').reset_index(drop=True)
    x = dfp['pos'].values
    y = dfp['AdaptIdx_blend'].values.astype(float)

    plt.figure()
    ax = plt.gca()

    # Fondo: franjas por bimestre (S1→S2→S3→…)
    _shade_bimonth_segments(ax, x, bimonth_markers, start_cycle_index=start_cycle_index)

    # Línea continua (sin cambiar color entre segmentos)
    plt.plot(x, y)  # una sola llamada => un solo color

    # Líneas de mes y bimestre
    if month_markers:
        for wk, _ in month_markers:
            plt.axvline(wk, color='gray', alpha=0.15)
        xs, labs = zip(*month_markers)
        plt.xticks(xs, labs, rotation=45, ha='right')
    if bimonth_markers:
        for wk, _ in bimonth_markers:
            plt.axvline(wk, color='black', linestyle=':', linewidth=1.2, alpha=0.7)

    # Guías y formato
    plt.axhline(1.0, color='gray', linestyle='--', alpha=0.5)
    plt.ylim(0, 1.2)
    plt.xlabel('Semanas'); plt.ylabel('Índice (0..1)')
    plt.title(f'Índice de adaptación (rolling 4m) — {title}')
    plt.tight_layout()
    os.makedirs(out_dir, exist_ok=True)
    plt.savefig(os.path.join(out_dir, f'adaptacion_IAN_{title}.png'))
    plt.close()

def group_chunks(df: pd.DataFrame, granularity='month', start_date_str=START_DATE):
        if granularity == 'month':
            for (y, m), g in df.groupby(['TX_YEAR','TX_MONTH'], sort=True):
                yield (int(y), int(m)), g, f"{int(y)}-{int(m):02d}"

        elif granularity == 'day':
            for (y, m, d), g in df.groupby(['TX_YEAR','TX_MONTH','TX_DAY'], sort=True):
                yield (int(y), int(m), int(d)), g, f"{int(y)}-{int(m):02d}-{int(d):02d}"

        elif granularity == 'week':
            d0 = pd.to_datetime(start_date_str)
            for wk, g in df.groupby('_week_idx', sort=True):
                wk = int(wk)
                wstart = d0 + pd.Timedelta(days=(wk-1)*7)
                iso = wstart.isocalendar()  # (year, week, weekday)
                label = f"{int(iso.year)}-W{int(iso.week):02d}"
                yield wk, g, label
        else:
            raise ValueError("granularity debe ser 'month' | 'week' | 'day'.")
        
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

def calculate_volatility_by_segment(metrics_df, seg_plateaux_df):
    
    metrics_indexed = metrics_df.set_index('pos')
    results = []
    
    for _, segment in seg_plateaux_df.iterrows():
        s, e, label = int(segment['start_pos']), int(segment['end_pos']), segment['label']
        
        # Filtra el F1 para las semanas DENTRO de este segmento
        f1_series = metrics_indexed.loc[s:e]['F1'].dropna()
        
        if len(f1_series) < 2:
            # No se puede calcular la varianza de 0 o 1 punto
            volatility = np.nan 
        else:
            mean_val = f1_series.mean()
            std_val = f1_series.std()
            # Calculamos la Desviación Estándar (medida de volatilidad)
            if mean_val > 0.001:
                volatility = std_val / mean_val
            else:
                volatility = 0.0
            
        results.append({'label': label, 'volatility': volatility})
        
    return pd.DataFrame(results)

def plot_volatility_chart(volatility_df, out_dir, title):
    """
    Genera un gráfico de barras para la volatilidad.
    """
    if volatility_df.empty:
        return
        
    plt.figure(figsize=(10, 6)) # Tamaño más grande para labels largos
    
    plt.bar(volatility_df['label'], volatility_df['volatility'])
    
    plt.title(f"Volatilidad del F1 por Patrón ({title})")
    plt.ylabel("Volatilidad (Desv. Estándar / Media)")
    plt.xlabel("Segmento de Patrón")
    plt.xticks(rotation=45, ha='right') # Rotamos labels para que no se solapen
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'volatilidad_f1_{title}.png'))
    plt.close()

def calculate_stable_recovery_by_segment(metrics_df, seg_plateaux_df, 
                                           recovery_threshold_pct=0.9, 
                                           stability_window_k=3):
   
    metrics_indexed = metrics_df.set_index('pos')
    results = []

    for _, segment in seg_plateaux_df.iterrows():
        s, e, label = int(segment['start_pos']), int(segment['end_pos']), segment['label']
        f1_plateau = segment['plateau_local'] # F1 esperado (calculado por tu función)
        
        # Valor por defecto: np.nan significa "Nunca se recuperó"
        recovery_weeks = np.nan 
        
        # Si no hay F1 esperado, o el segmento es muy corto, no podemos medir
        if not np.isfinite(f1_plateau) or f1_plateau <= 0:
            results.append({'label': label, 'recovery_weeks': recovery_weeks})
            continue
            
        f1_threshold = f1_plateau * recovery_threshold_pct
        f1_series = metrics_indexed.loc[s:e]['F1'].values # F1s solo de este segmento
        
        if len(f1_series) < stability_window_k:
            results.append({'label': label, 'recovery_weeks': recovery_weeks})
            continue

        # Buscamos la primera semana 'i' donde las K semanas (i a i+K-1)
        # están todas por encima del umbral.
        for i in range(len(f1_series) - stability_window_k + 1):
            window = f1_series[i : i + stability_window_k]
            
            # np.all() comprueba si TODOS los valores en la ventana son True
            if np.all(window >= f1_threshold):
                recovery_weeks = i # 0-indexed: 0 = recuperado en la 1ra sem
                break # ¡Encontrado!
        
        results.append({'label': label, 'recovery_weeks': recovery_weeks})
        
    return pd.DataFrame(results)

def plot_recovery_chart(recovery_df, out_dir, title, stability_window_k=3):
    
    if recovery_df.empty:
        return
        
    plt.figure(figsize=(10, 6))
    
    plot_df = recovery_df.copy()
    

    max_real_recovery = plot_df['recovery_weeks'].max()
    if not np.isfinite(max_real_recovery) or max_real_recovery == 0:
        max_real_recovery = 8 # Un valor base si todos fallaron o fueron 0
  
    failure_value = max_real_recovery * 1.1 + 1 
    plot_df['plot_value'] = plot_df['recovery_weeks'].fillna(failure_value)
    
    plot_df['plot_value'] = plot_df['plot_value'].apply(lambda x: 0.1 if x == 0 else x)

    # 4. Creamos las barras
    bars = plt.bar(plot_df['label'], plot_df['plot_value'])
    

    legend_labels = {}
    
    for i, bar in enumerate(bars):
        valor_real = plot_df['recovery_weeks'].iloc[i]
        
        if not np.isfinite(valor_real):
            # Caso 1: Fallo (NaN)
            bar.set_color('red')
            bar.set_edgecolor('black')
            legend_labels['Nunca se recuperó (NaN)'] = bar
            
        elif valor_real == 0:
            # Caso 2: Éxito Instantáneo (0)
            bar.set_color('green')
            legend_labels['Éxito Instantáneo (0 Semanas)'] = bar
            
        else:
            # Caso 3: Éxito Normal (> 0)
            # (la barra ya es azul por defecto)
            legend_labels['Recuperación Exitosa'] = bar

    plt.title(f"Tiempo de Recuperación Estable (K=5) ({title})")
    plt.ylabel(f"Semanas para Recuperación (K=5)")
    plt.xlabel("Segmento de Patrón")
    plt.xticks(rotation=45, ha='right')
  
    if legend_labels:
         plt.legend(legend_labels.values(), legend_labels.keys())

    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'recuperacion_f1_{title}.png'))
    plt.close()

In [ ]:
# =======================
# CL con Avalanche: ER y SI
# =======================
import os, time, math
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import average_precision_score, precision_recall_curve, f1_score


from avalanche.training import Naive   
from avalanche.training.plugins import ReplayPlugin, SynapticIntelligencePlugin,MIRPlugin
from avalanche.benchmarks.utils import AvalancheDataset
from avalanche.benchmarks.scenarios.dataset_scenario import benchmark_from_datasets


# -----------------------
# Modelo MLP tabular
# -----------------------
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(128, 64), pdrop=0.1):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(pdrop)]
            d = h
        layers += [nn.Linear(d, 1)]  # logit binario
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x).squeeze(1)  # (N,)

# -----------------------
# Utils: DF -> TensorDataset/DataLoader
# -----------------------
def df_to_tensors(df, features, target='TX_FRAUD', scaler=None, device='cpu'):
    X = df[features].astype('float32').values
    y = df[target].astype('float32').values
    if scaler is not None:
        X = scaler.transform(X)
    X_t = torch.from_numpy(X).to(device)
    y_t = torch.from_numpy(y).to(device)
    return TensorDataset(X_t, y_t)

def make_loader(dataset, batch_size=1024, shuffle=True):
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=False)

# -----------------------
# Pretraining + calibración umbral (F1)
# -----------------------
def pretrain_calibrate(train_df, val_df, features, device='cuda', 
                       hidden=(128,64), pdrop=0.1,
                       lr=1e-3, epochs=3, batch_size=2048):
    # Escalador (tabular)
    scaler = StandardScaler().fit(train_df[features].astype('float32').values)

    # Datos
    tr_ds = df_to_tensors(train_df, features, scaler=scaler, device=device)
    va_ds = df_to_tensors(val_df,   features, scaler=scaler, device=device)
    tr_ld = make_loader(tr_ds, batch_size=batch_size, shuffle=True)
    va_ld = make_loader(va_ds, batch_size=4096, shuffle=False)

    # Modelo
    model = MLP(len(features), hidden=hidden, pdrop=pdrop).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    crit = nn.BCEWithLogitsLoss()

    # Entrenamiento simple
    model.train()
    for _ in range(epochs):
        for xb, yb in tr_ld:
            opt.zero_grad()
            logits = model(xb)
            loss = crit(logits, yb)
            loss.backward()
            opt.step()

    # Calibración de umbral en val (F1)
    model.eval()
    with torch.no_grad():
        scores = []
        y_true = []
        for xb, yb in va_ld:
            logits = model(xb)
            s = torch.sigmoid(logits).detach().cpu().numpy()
            scores.append(s)
            y_true.append(yb.detach().cpu().numpy())
    pv = np.concatenate(scores)
    yv = np.concatenate(y_true).astype(int)

    ap_ref = float(average_precision_score(yv, pv)) if yv.sum() > 0 else float('nan')
    prec, rec, thr = precision_recall_curve(yv, pv)
    f1s = (2*prec*rec)/(prec+rec+1e-12)
    if len(thr) > 0:
        best_idx = int(np.nanargmax(f1s[:-1]))
        thr_star = float(thr[best_idx])
        f1_ref = float(f1s[best_idx])
    else:
        thr_star, f1_ref = 0.5, float('nan')

    return model, scaler, thr_star, ap_ref, f1_ref

# -----------------------
# Construir benchmark semanal desde tu eval_df
# (usa tu group_chunks y crea listas de datasets por semana)
# -----------------------
def weekly_benchmark_from_df(eval_df, features, scaler, target='TX_FRAUD'):
    train_datasets, test_datasets = [], []
    week_keys, week_labels = [], []

    for key, chunk, label in group_chunks(eval_df, granularity='week', start_date_str=START_DATE):
        ds = df_to_avalanche_tabular(chunk, features, target=target, scaler=scaler, task_label=0)
        train_datasets.append(ds)   # entreno en la semana t
        test_datasets.append(ds)    # evalúo en la semana t (post-train)
        week_keys.append(int(key))
        week_labels.append(label)

    bm = benchmark_from_datasets(train=train_datasets, test=test_datasets)
    return bm, week_keys, week_labels

# -----------------------
# Crear estrategia ER / SI en Avalanche
# -----------------------
def make_strategy(strategy_name, model, device='cuda', lr=1e-3, 
                  si_lambda=0.1, memory_size=100000, 
                  train_mb_size=1024, eval_mb_size=4096,
                  train_epochs=3,
                  mir_batch_size_mem=512,
                  mir_subsample=4096
                 ):
    crit = nn.BCEWithLogitsLoss()
    opt  = torch.optim.Adam(model.parameters(), lr=lr)


    plugins = []
    name = strategy_name.lower()
    
    if name in ('er','replay'):
        # Añadimos el PLUGIN de Replay
        plugins.append(
            MIRPlugin(
                mem_size=int(100000),
                batch_size_mem=int(mir_batch_size_mem),
                subsample=int(4096)
            )
        )
    elif name in ('si','synaptic','synapticintelligence'):
      
        plugins.append(
            SynapticIntelligencePlugin(
                si_lambda=float(0.5)
            )
        )
    else:
        raise ValueError(f"EstrategIA CL no soportada: {strategy_name}")

    cl_strategy = Naive(
        model=model, 
        optimizer=opt, 
        criterion=crit,
        train_mb_size=train_mb_size, 
        eval_mb_size=eval_mb_size, 
        device=device,
        plugins=plugins 
    )
    
    return cl_strategy

# -----------------------
# Inferencia con tiempos y métricas
# -----------------------
@torch.no_grad()
def evaluate_week(model, dataset, thr=0.5, device='cuda', eval_mb_size=4096):
    """
    Devuelve: AUPRC, F1, Recall (sensibilidad), Specificity, G-Mean,
              infer_ms_per_tx (TIEMPO TOTAL de la semana, en ms, por compatibilidad),
              infer_ms_avg (tiempo promedio por transacción, en ms).
    """
    import time, numpy as np
    from torch.utils.data import DataLoader
    from sklearn.metrics import (
        average_precision_score, f1_score, recall_score, precision_score
    )

    model.eval()
    loader = DataLoader(dataset, batch_size=eval_mb_size, shuffle=False)

    tot_s = 0.0
    s_all, y_all = [], []

    for batch in loader:
        # AvalancheDataset puede devolver (x, y) o (x, y, task)
        if isinstance(batch, (list, tuple)) and len(batch) == 3:
            xb, yb, _ = batch
        else:
            xb, yb = batch

        t0 = time.perf_counter()
        logits = model(xb.to(device))
        tot_s += time.perf_counter() - t0

        s = torch.sigmoid(logits).detach().cpu().numpy()
        y = yb.detach().cpu().numpy()
        s_all.append(s); y_all.append(y)

    s_all = np.concatenate(s_all)
    y_all = np.concatenate(y_all).astype(int)

    # --- Métricas ---
    ap  = float(average_precision_score(y_all, s_all)) if y_all.sum() > 0 else float('nan')
    yhat = (s_all >= float(thr)).astype(int)

    f1   = float(f1_score(y_all, yhat, zero_division=0))
    rec  = float(recall_score(y_all, yhat, pos_label=1, zero_division=0))  # sensibilidad
    spec = float(recall_score(y_all, yhat, pos_label=0, zero_division=0))  # especificidad
    prec = float(precision_score(y_all, yhat, zero_division=0))
    try:
        gmean = float((rec * spec) ** 0.5)
    except Exception:
        gmean = float('nan')

    # --- Latencias ---
    infer_total_ms_chunk = tot_s * 1e3
    infer_ms_avg         = (tot_s / max(1, len(y_all))) * 1e3

   
    return {
        "AUPRC": ap,
        "F1": f1,
        "Recall": rec,
        "Specificity": spec,
        "Precision": prec,
        "G-Mean": gmean,
        "infer_ms_per_tx": infer_total_ms_chunk,  
        "infer_ms_avg": infer_ms_avg            
    }

# -----------------------
# Experimento CL (ER o SI)
# -----------------------
def run_cl_model_ER_or_SI(
        df, features,
        pre_train_df, pre_val_df,
        eval_df,
        base_week=None,
        out_dir='./OUT_CL',
        strategy_name='er',
        device='cuda',
        # HPR:
        hidden=(128,64), pdrop=0.1, lr=1e-3,
        epochs_pre=3, batch_size=2048,
        si_lambda=0.1, memory_size=100000,
        eval_mb_size=4096
    ):
    import os, time
    import numpy as np, pandas as pd
    os.makedirs(out_dir, exist_ok=True)

    # === 1) Pretraining + calibración umbral (reusa tu pretrain_calibrate) ===
    model, scaler, thr, ap_ref, f1_ref = pretrain_calibrate(
        pre_train_df, pre_val_df, features,
        device=device, hidden=hidden, pdrop=pdrop,
        lr=lr, epochs=epochs_pre, batch_size=batch_size
    )

    # === 2) Benchmark semanal (tabular) ===
    bench, week_keys, week_labels = weekly_benchmark_from_df(
        eval_df, features, scaler=scaler, target='TX_FRAUD'
    )

    # === 3) Estrategia (Naive + Plugin) ===
    strategy = make_strategy(
        strategy_name, model, device=device, lr=lr,
        si_lambda=si_lambda, memory_size=memory_size,
        train_mb_size=batch_size, eval_mb_size=eval_mb_size
    )

    rows, train_time_rows = [], []

    for exp_idx, (train_exp, test_exp) in enumerate(zip(bench.train_stream, bench.test_stream), start=1):
        key   = week_keys[exp_idx-1]
        label = week_labels[exp_idx-1]
        pos   = (int(key) - int(base_week) + 1) if base_week is not None else int(key)

        # --- ENTRENAR ---
        t0 = time.perf_counter()
        strategy.train(train_exp)
        train_time_s = time.perf_counter() - t0
        train_time_rows.append({"pos": pos, "label": label, "train_time_s": train_time_s})

        # --- EVALUAR (post-train) ---
        m = evaluate_week(
            strategy.model, test_exp.dataset, thr=thr, device=device, eval_mb_size=eval_mb_size
        )

        rows.append({
            "pos": pos,
            "label": label,
            "n": len(test_exp.dataset),
            **m  # AUPRC, F1, Recall, Specificity, Precision, G-Mean, infer_ms_per_tx, infer_ms_avg
        })

    metrics     = pd.DataFrame(rows).sort_values('pos').reset_index(drop=True)
    train_times = pd.DataFrame(train_time_rows).sort_values('pos').reset_index(drop=True)

    metrics.to_csv(os.path.join(out_dir, f'metrics_{strategy_name.upper()}.csv'), index=False)
    train_times.to_csv(os.path.join(out_dir, f'train_times_{strategy_name.upper()}.csv'), index=False)
    return metrics, train_times, {"thr": thr, "ap_ref": ap_ref, "f1_ref": f1_ref}


import torch
from torch.utils.data import TensorDataset

def df_to_avalanche_tabular(df, features, target='TX_FRAUD', scaler=None, task_label=0):
    """
    Devuelve un AvalancheDataset válido para benchmark_from_datasets:
      - cada sample: (x, y_float)  -> útil para BCEWithLogitsLoss
      - .targets (int64)           -> lo que Avalanche/Replay necesitan
      - .task_labels (int64)       -> por compatibilidad (todo a 0)
    """
    from avalanche.benchmarks.utils import as_classification_dataset
    X = df[features].astype('float32').values
    if scaler is not None:
        X = scaler.transform(X)
    y_float = df[target].astype('float32').values  # 0/1 en float

    base = TabularBceDatasetWithTargets(X, y_float)

    avl_ds = as_classification_dataset(base)   
   

    return avl_ds


import torch
from torch.utils.data import Dataset

class TabularBceDatasetWithTargets(Dataset):
    """
    __getitem__ -> (x, y_float) para BCEWithLogitsLoss
    .targets    -> int64 (0/1), que Avalanche/Replay usan internamente
    """
    def __init__(self, X_np, y_np_float):
        X_t = torch.as_tensor(X_np, dtype=torch.float32)
        y_f = torch.as_tensor(y_np_float, dtype=torch.float32)
        self.X = X_t
        self.y = y_f
        # atributo que Avalanche espera encontrar:
        self._targets = (y_f > 0.5).to(torch.int64)

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]   # y en float para BCE

    @property
    def targets(self):
        # Avalanche mira este atributo; tensor va bien
        return self._targets

c:\Users\USER\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
schedule_ym = [
  # ======================
  # PRETRAIN (Ene–Abr 2025): S1 + S2 simultáneos
  # ======================
  {"scenario": 1, "start": {"year": 2025, "month": 1}, "end": {"year": 2025, "month": 4},
   "params": {"amount_threshold": 140}},

  {"scenario": 2, "start": {"year": 2025, "month": 1}, "end": {"year": 2025, "month": 4},
   "params": {"n_per_day": 4, "window_days": 60}},



  # ======================
  # Bimestres (S1 → S2 → S3) hasta 24 meses
  # ======================

  # Bloque 1 (May–Jun 2025): S1
  {"scenario": 1, "start": {"year": 2025, "month": 5}, "end": {"year": 2025, "month": 6},
   "params": {"amount_threshold": 120}},

  # Bloque 2 (Jul–Ago 2025): S2
  {"scenario": 2, "start": {"year": 2025, "month": 7}, "end": {"year": 2025, "month": 8},
   "params": {"n_per_day": 4, "window_days": 62}},

  # Bloque 2 (Jul–Ago 2025): S2
  #{"scenario": 2, "start": {"year": 2025, "month": 8}, "end": {"year": 2025, "month": 8},
  # "params": {"n_per_day": 4, "window_days": 28}},

  # Bloque 3 (Sep 2025): S3 (más suave)
  {"scenario": 3, "start": {"year": 2025, "month": 9}, "end": {"year": 2025, "month": 10},
   "params": {"n_customers_per_day": 15, "window_days": 60, "amp_factor": 6, "frac_to_flip": 1/2}},
  # Bloque 3 (Oct 2025): S3 (más intenso)
  #{"scenario": 3, "start": {"year": 2025, "month": 10}, "end": {"year": 2025, "month": 10},
  # "params": {"n_customers_per_day": 15, "window_days": 30, "amp_factor": 6, "frac_to_flip": 1/2}},

  # Bloque 4 (Nov–Dic 2025, NAVIDAD): S1 “suavizado”
  {"scenario": 1, "start": {"year": 2025, "month": 11}, "end": {"year": 2025, "month": 12},
   "params": {"amount_threshold": 150}},
 # {"scenario": 1, "start": {"year": 2025, "month": 12}, "end": {"year": 2025, "month": 12},
   #"params": {"amount_threshold": 180}},

  # Bloque 5 (Ene–Feb 2026): S2
  {"scenario": 2, "start": {"year": 2026, "month": 1}, "end": {"year": 2026, "month": 2},
   "params": {"n_per_day": 4, "window_days": 59}},

   #{"scenario": 2, "start": {"year": 2026, "month": 2}, "end": {"year": 2026, "month": 2},
  # "params": {"n_per_day": 4, "window_days": 28}},

  # Bloque 6 (Mar–Abr 2026): S3
  {"scenario": 3, "start": {"year": 2026, "month": 3}, "end": {"year": 2026, "month": 4},
   "params": {"n_customers_per_day": 10, "window_days": 30, "amp_factor": 6, "frac_to_flip": 1/2}},

  # Bloque 7 (May–Jun 2026): S1
  {"scenario": 1, "start": {"year": 2026, "month": 5}, "end": {"year": 2026, "month": 6},
   "params": {"amount_threshold": 120}},

  # Bloque 8 (Jul–Ago 2026): S2
  {"scenario": 2, "start": {"year": 2026, "month": 7}, "end": {"year": 2026, "month": 8},
   "params": {"n_per_day": 4, "window_days": 62}},
   # Bloque 8 (Jul–Ago 2026): S2
 # {"scenario": 2, "start": {"year": 2026, "month": 8}, "end": {"year": 2026, "month": 8},
  # "params": {"n_per_day": 4, "window_days": 28}},

  # Bloque 9 (Sep–Oct 2026): S3
  {"scenario": 3, "start": {"year": 2026, "month": 9}, "end": {"year": 2026, "month": 10},
   "params": {"n_customers_per_day": 15, "window_days": 60, "amp_factor": 6, "frac_to_flip": 1/2}},
  # Bloque 9 (Oct 2026): S3 (más intenso)
  #{"scenario": 3, "start": {"year": 2026, "month": 10}, "end": {"year": 2026, "month": 10},
  # "params": {"n_customers_per_day": 15, "window_days": 30, "amp_factor": 6, "frac_to_flip": 1/2}},

  # Bloque 10 (Nov–Dic 2026, NAVIDAD): S1 “suavizado”
  {"scenario": 1, "start": {"year": 2026, "month": 11}, "end": {"year": 2026, "month": 12},
   "params": {"amount_threshold": 150}},
 # {"scenario": 1, "start": {"year": 2026, "month": 12}, "end": {"year": 2026, "month": 12},
  # "params": {"amount_threshold": 180}},   # puedes subir a 180 en dic si quieres aún más “suavizado”
]

In [ ]:

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Dispositivo encontrado:", device)
# 1) Cargar y preparar
df = load_all_parquet_by_year(BASE_PATH)
df = add_time_indexes(df)               # ya añade _week_idx, etc.


print("Dataset Cargado")


train_mask, val_mask = split_pretrain_val(df, pre_months=4, val_days_last_month=VAL_DAYS_LAST_MONTH)
pre_train_df = df[train_mask]
pre_val_df   = df[val_mask]
eval_df      = df[df['_month_idx'] >= 5]

print("Se realizo el split de preentrenamiento y evaluación.")

base_week = compute_base_week(eval_df)
month_marks = compute_month_markers(eval_df, base_week)
bimonth_marks = compute_bimonth_markers(eval_df, pre_months=4, base_week=base_week)

EXTRA_FEATS = [
    'term_tx_cum', 'term_frd_cum', 'term_fraud_rate_cum',
    'term_tx_ewm', 'term_time_since_prev'
]
features = [c for c in (FEATURES + EXTRA_FEATS) if c in df.columns]

df[features] = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
df['TX_FRAUD'] = df['TX_FRAUD'].astype('float32')

print("Listo para ejecutar CL con características:", features)
# 3) Ejecutar ER
metrics_er, train_er, info_er = run_cl_model_ER_or_SI(
    df, features, pre_train_df, pre_val_df, eval_df,
    base_week=base_week, out_dir=os.path.join(OUT_DIR, "ER_CL"),
    strategy_name='er', device=device, memory_size=100000,
    hidden=(128,64), pdrop=0.1, lr=1e-3, epochs_pre=3, batch_size=2048
)
print("Se ejecuto el CL con ER.")
# 4) Ejecutar SI
metrics_si, train_si, info_si = run_cl_model_ER_or_SI(
    df, features, pre_train_df, pre_val_df, eval_df,
    base_week=base_week, out_dir=os.path.join(OUT_DIR, "SI_CL"),
    strategy_name='si', device=device, si_lambda=0.5,
    hidden=(128,64), pdrop=0.1, lr=1e-3, epochs_pre=3, batch_size=2048
)
print("Se ejecuto el CL con SI.")

for name, metrics ,train in [('ER', metrics_er,train_er), ('SI', metrics_si,train_si)]:
    out = os.path.join(OUT_DIR, f"{name}_CL")
    # Series AUPRC/F1/latencia
    plot_series(metrics,train, out, month_markers=month_marks, bimonth_markers=bimonth_marks,title=f"{name.upper()}")


    segments_ym = schedule_to_week_segments_with_scenario(schedule_ym, eval_df, base_week)
    seg_plateaux = compute_segment_plateaux(metrics, segments_ym, window=8, method='percentile', q=0.80)


    print(f"Calculando métricas de adaptación para {name.upper()}...")
    
 
    volatility_df = calculate_volatility_by_segment(metrics, seg_plateaux)
    
    
    #volatility_df.to_csv(os.path.join(model_dir, 'volatilidad_por_patron.csv'), index=False)
    plot_volatility_chart(volatility_df, out, title=f"BASELINE 2 {name.upper()}")

    K_ESTABILIDAD =5 
    PCT_RECUPERACION = 1
    
    recovery_df = calculate_stable_recovery_by_segment(
        metrics, 
        seg_plateaux,
        recovery_threshold_pct=PCT_RECUPERACION,
        stability_window_k=K_ESTABILIDAD
    )
    
    # Guardamos los datos y el gráfico
    #recovery_df.to_csv(os.path.join(model_dir, 'recuperacion_por_patron.csv'), index=False)
    plot_recovery_chart(recovery_df, out, 
                        title=f"BASELINE 2 {name.upper()}", 
                        stability_window_k=K_ESTABILIDAD)


Dispositivo encontrado: cuda
Dataset Cargado
Se realizo el split de preentrenamiento y evaluación.
Semana base: 18, Fecha: 2025-05-01
Listo para ejecutar CL con características: ['TX_AMOUNT', 'TX_TIME_DAYS', 'TX_TIME_SECONDS', 'x_customer_id', 'y_customer_id', 'mean_amount', 'std_amount', 'mean_nb_tx_per_day', 'x_terminal_id', 'y_terminal_id']
-- >> Start of training phase << --
100%|██████████| 9/9 [00:00<00:00, 70.50it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream = 0.0404
	Top1_Acc_Epoch/train_phase/train_stream = 0.0000
-- >> End of training phase << --


/home/amalpartida/entorno_angel/lib/python3.12/site-packages/avalanche/training/plugins/mir.py:111: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)


-- >> Start of training phase << --
100%|██████████| 10/10 [00:00<00:00, 28.76it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream = 0.0287
	Top1_Acc_Epoch/train_phase/train_stream = 0.0000
-- >> End of training phase << --


/home/amalpartida/entorno_angel/lib/python3.12/site-packages/avalanche/training/plugins/mir.py:111: DeprecationWarning: Call to deprecated function update (removal in version 0.7: switch to pre_adapt and post_adapt)
  self.storage_policy.update(strategy, **kwargs)


-- >> Start of training phase << --
100%|██████████| 10/10 [00:00<00:00, 69.34it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream = 0.0235
	Top1_Acc_Epoch/train_phase/train_stream = 0.0000
-- >> End of training phase << --


...

/home/amalpartida/entorno_angel/lib/python3.12/site-packages/avalanche/training/plugins/synaptic_intelligence.py:316: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  np.square(


-- >> Start of training phase << --
100%|██████████| 14/14 [00:00<00:00, 21.69it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream = 0.0213
	Top1_Acc_Epoch/train_phase/train_stream = 0.0000
-- >> End of training phase << --


/home/amalpartida/entorno_angel/lib/python3.12/site-packages/avalanche/training/plugins/synaptic_intelligence.py:316: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  np.square(


-- >> Start of training phase << --
100%|██████████| 4/4 [00:00<00:00, 22.28it/s]
Epoch 0 ended.
	Loss_Epoch/train_phase/train_stream = 0.0191
	Top1_Acc_Epoch/train_phase/train_stream = 0.0000
-- >> End of training phase << --
Se ejecuto el CL con SI.


/home/amalpartida/entorno_angel/lib/python3.12/site-packages/avalanche/training/plugins/synaptic_intelligence.py:316: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  np.square(
